# Práctica 2 · Paso 3 — Medidas de reducción del riesgo, adaptación y mitigación

**Cambio Climático · Ingeniería en Riesgos de Desastres · ESPAM MFL** · Material CC-IRD-P2-03 · Versión 1.0 · noviembre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-2/RD2_03_matriz_medidas.ipynb)

**Objetivo.** Proponer, comparar y priorizar medidas para el cantón de mayor riesgo de su provincia, **diferenciando** lo que hace la adaptación (reduce el riesgo local) de lo que hace la mitigación (reduce las emisiones), y ubicando cada medida en el marco normativo: Acuerdo de París y Segunda NDC, PLANACC y Marco de Sendai.

**Producto.** Matriz de evaluación multicriterio, ranking con prueba de robustez, gráfico efectividad–factibilidad, tabla del marco normativo (NDC, PLANACC, Sendai y actor responsable), cartera priorizada y **riesgo residual**: cuánto bajaría el riesgo del cantón si se aplicara la cartera. Es la última pieza del borrador del capítulo 2 (**evaluación formativa, 2 puntos**).

**Requisito.** Haber ejecutado `RD2_01_exposicion` y `RD2_02_vulnerabilidad` con la misma provincia, con la planilla de vulnerabilidad completa (no provisional).

**Guía para el estudiante.** Paso a paso, ejemplo de evaluación y cómo llevar cada salida al capítulo 2: [`guia_RD2_03.md`](https://github.com/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-2/guia_RD2_03.md).

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Ejecute hasta la sección 5: el cuaderno le sugiere medidas del catálogo según el riesgo de su cantón y le imprime un bloque para copiar. Péguelo en la celda de la sección 6, quédese con 6 a 10 medidas (al menos una de **mitigación pura**), califíquelas de 1 a 5 **con su justificación** y ejecute el resto. Tiempo estimado: 1 hora en clase y 2 horas sin docente.

**Con energía o internet intermitentes.** La evaluación (sección 6) se puede preparar en papel o en un archivo de texto con el formato del bloque, y pegarse cuando haya conexión; el cuaderno tarda menos de 1 minuto.

## 1. Datos del equipo

In [ ]:
# @title Equipo, provincia y cantón de trabajo
ESTUDIANTES = "Nombre Apellido · Nombre Apellido"  # @param {type:"string"}
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
CANTON_FOCO = ""  # @param {type:"string"}
# Déjelo VACÍO para trabajar con el cantón de mayor riesgo de RD2_02 (lo normal).
# Escriba otro cantón prioritario solo si el equipo lo justifica (por ejemplo, el segundo con riesgo casi igual y mejores datos).
print(f"Equipo: {ESTUDIANTES} · provincia de {PROVINCIA}")

## 2. Entorno de trabajo y productos de RD2_01 y RD2_02

In [ ]:
import os, sys, pathlib, subprocess, datetime, json, re, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, riesgo as RG, estrategias as ES, vulnerabilidad as VU

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(os.environ.get("CC_BASE", "."))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90)
SP = limites.slug(PROVINCIA)
RES = BASE / "CC_Riesgos" / "practica-2" / SP
NECESARIOS = [RES / "RD2_02_metadatos.json", RES / f"RD2_02_riesgo_{SP}.csv", RES / f"RD2_02_calificaciones_{SP}.csv",
              RES / f"RD2_01_puntajes_exposicion_{SP}.csv"]
faltan = [p.name for p in NECESARIOS if not p.exists()]
if faltan:
    raise FileNotFoundError(f"Faltan productos en {RES}: {faltan}.\nEjecute RD2_01 y RD2_02 con la provincia {PROVINCIA}.")
META2 = json.loads(NECESARIOS[0].read_text(encoding="utf-8"))
RIESGO = pd.read_csv(NECESARIOS[1])
CAL = pd.read_csv(NECESARIOS[2])
PUNT = pd.read_csv(NECESARIOS[3], dtype={"codigo": str})
A = META2["amenazas"]
if META2.get("provisional"):
    print("⚠ RD2_02 está marcado como PROVISIONAL (planilla de vulnerabilidad incompleta).\n"
          "  Puede practicar, pero complete la planilla y vuelva a ejecutar RD2_02 antes de entregar.\n")
FOCO = CANTON_FOCO.strip() or META2["canton_critico"]
_norm = {limites.normalizar(c): c for c in RIESGO.canton.unique()}
if limites.normalizar(FOCO) not in _norm:
    raise ValueError(f"«{FOCO}» no está entre los cantones evaluados en RD2_02: {', '.join(RIESGO.canton.unique())}")
FOCO = _norm[limites.normalizar(FOCO)]
R_FOCO = RIESGO[RIESGO.canton == FOCO].sort_values("R", ascending=False)
AMEN_REL = R_FOCO.loc[R_FOCO.R >= 2.6, "clave"].tolist() or R_FOCO.clave.tolist()[:1]
C_FOCO = CAL[CAL.canton == FOCO]
FACT = (C_FOCO[(C_FOCO.componente == "sensibilidad") & (C_FOCO.valor >= 4)].clave.tolist() +
        C_FOCO[(C_FOCO.componente == "capacidad") & (C_FOCO.valor <= 2)].clave.tolist())
print(f"Cantón de trabajo: {FOCO}" + (" (el de mayor riesgo en RD2_02)" if not CANTON_FOCO.strip() else " (elegido por el equipo)"))

## 3. Punto de partida: el riesgo del cantón

Las medidas deben responder a **lo que salió en su análisis**: las amenazas con riesgo medio o mayor y los indicadores críticos de la vulnerabilidad (sensibilidad 4–5 o capacidad 1–2). Una medida que no responde a ninguno de ellos necesita una justificación muy clara.

In [ ]:
display(R_FOCO[["amenaza", "A", "E", "V", "R", "clase", "pesa_mas"]].set_index("amenaza"))
print("Amenazas de riesgo medio o mayor:", ", ".join(RG.AMENAZA_CORTA[a] for a in AMEN_REL))
TAB_C = C_FOCO.assign(indicador=C_FOCO.clave.map({**VU.SENSIBILIDAD, **VU.CAPACIDAD}))[["componente", "indicador", "valor", "dato_encontrado", "fuente"]]
TAB_C["crítico"] = np.where(C_FOCO.clave.isin(FACT), "◀ crítico", "")
display(TAB_C.set_index("indicador"))

## 4. Mitigación, adaptación y reducción del riesgo: qué hace cada una

Las dos son necesarias y **no se reemplazan**. La pregunta del RA B es justamente esa: cuál es el alcance de cada una y cuándo es pertinente.

In [ ]:
display(VU.ALCANCE)

## 5. Medidas sugeridas por su análisis

El cuaderno busca en el catálogo del curso (`herramientas/estrategias.py`, 29 medidas tomadas del IPCC AR6, la Segunda NDC y el PLANACC) las que atienden **las amenazas** y **los indicadores críticos** de su cantón. Tipo: **A** adaptación o reducción del riesgo · **M** mitigación · **MA** las dos. La columna `reduce` dice qué componente baja cada medida.

Además se listan las medidas de **mitigación pura (M)**: no salen de su análisis de riesgo, porque no reducen el riesgo local; elija al menos una que tenga sentido en su provincia (por ejemplo, transporte en una ciudad grande, residuos en un cantón con botadero, ganadería en una zona ganadera) para poder compararla con las de adaptación.

In [ ]:
CAT = ES.catalogo()
CAT["reduce"] = VU.componentes_que_reduce(CAT)
SUG = ES.sugerir([], AMEN_REL, FACT)
SUG["reduce"] = CAT.loc[SUG.index, "reduce"]
SUG["n_motivos"] = SUG.motivo.str.count(";") + 1 + SUG.motivo.str.count(",")
SUG = SUG.sort_values(["n_motivos", "tipo"], ascending=[False, True])
print(f"Medidas que responden al riesgo de {FOCO} ({len(SUG)}):")
display(SUG[["opcion", "tipo", "reduce", "motivo", "sector"]])
MIT = CAT[CAT.tipo == "M"]
print("Medidas de mitigación pura (elija al menos una):")
display(MIT[["opcion", "reduce", "sector", "alerta_mala_adaptacion"]])
print("# ---- Copie este bloque a la celda de la sección 6. Deje entre 6 y 10 líneas: al menos 3 de tipo A o MA,")
print("# ---- al menos 2 de tipo M o MA, y al menos una M pura (quite el # de su línea). Cambie los 3 por su calificación.")
print("# id; efectividad; costo; factibilidad; cobeneficios; sinergia; mala_adaptacion; alineacion; justificación")
BLOQUE = pd.concat([SUG[SUG.tipo == "A"].head(5), SUG[SUG.tipo == "MA"].head(2)])
def _corto(txt, n=55):
    return txt if len(txt) <= n else txt[:n].rsplit(" ", 1)[0].rstrip(",:(") + "…"
for i, r in BLOQUE.iterrows():
    print(f"{i}; 3; 3; 3; 3; {4 if r.tipo == 'MA' else 2}; 3; 3; {_corto(r.opcion)}: justifique aquí")
for i, r in MIT.iterrows():
    print(f"# {i}; 2; 3; 3; 3; 1; 3; 3; {_corto(r.opcion)}: justifique aquí")

## 6. Evaluación multicriterio (1–5; en todos, 5 es lo mejor)

| Criterio | Pregunta para su cantón | 1 | 5 |
|---|---|---|---|
| efectividad | Medida A: ¿cuánto reduce el riesgo del cantón? Medida M: ¿cuánto reduce las emisiones de la provincia? | casi nada | mucho, en la amenaza o el indicador crítico |
| costo | ¿Cuánto cuesta construirla y mantenerla para el GAD? | muy cara | barata |
| factibilidad | ¿Hay competencias, normas, actores y experiencia en el cantón? | no hay | hay, y ya se hizo algo parecido |
| cobeneficios | ¿Trae otros beneficios (salud, empleo, agua, biodiversidad)? | ninguno | muchos y claros |
| sinergia | ¿Sirve a la vez para mitigar y para adaptarse? | solo una, o en conflicto | las dos, claramente |
| mala_adaptacion | ¿Puede trasladar el riesgo a otros o crear un riesgo nuevo? | riesgo alto | riesgo bajo |
| alineacion | ¿Está en la Segunda NDC, el PLANACC o el PDOT del cantón? | no aparece | aparece como prioridad |

**Cómo escribir cada línea:** `id; efectividad; costo; factibilidad; cobeneficios; sinergia; mala_adaptacion; alineacion; justificación`. La justificación es **una frase con un dato**: por qué ese id, para qué amenaza o indicador, y qué la hace factible o la frena. Ejemplo:

`A01; 5; 4; 4; 3; 1; 5; 5; Chone tiene alerta 2 en RD2_02, el río Chone se desborda cada invierno y la SNGR ya opera alertas en la provincia.`

**No use punto y coma dentro de la justificación**: el punto y coma separa las columnas. Use comas.

In [ ]:
# @title Pesos de los criterios (se normalizan para sumar 1)
W_EFECTIVIDAD = 0.25  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_COSTO = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_FACTIBILIDAD = 0.20  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_COBENEFICIOS = 0.15  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_SINERGIA = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_MALA_ADAPTACION = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_ALINEACION = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
JUSTIFICACION_PESOS = ""  # @param {type:"string"}
# Ejemplo: «Efectividad y factibilidad pesan más porque el GAD necesita medidas que funcionen antes del próximo invierno».

In [ ]:
# @title Evaluación: pegue aquí el bloque de la sección 5 y edítelo
EVALUACION = """
"""

## 7. Matriz, ranking y robustez

In [ ]:
W = {"efectividad": W_EFECTIVIDAD, "costo": W_COSTO, "factibilidad": W_FACTIBILIDAD, "cobeneficios": W_COBENEFICIOS,
     "sinergia": W_SINERGIA, "mala_adaptacion": W_MALA_ADAPTACION, "alineacion": W_ALINEACION}
sw = sum(W.values()) or 1
if abs(sw - 1) > 0.01: print(f"Los pesos sumaban {sw:.2f}; se normalizaron a 1.")
W = {k: v / sw for k, v in W.items()}
POR_DEFECTO = not EVALUACION.strip() or all(l.strip().startswith("#") for l in EVALUACION.strip().splitlines())
if POR_DEFECTO:
    print("⚠ La celda de evaluación está vacía: se evalúan las sugerencias con 3 y una medida M. NO es válido para entregar.\n")
    ids = list(BLOQUE.index) + [MIT.index[0]]
    EV = pd.DataFrame({k: 3 for k in ES.CRITERIOS}, index=ids); EV["justificacion"] = ""
else:
    EV = ES.leer_evaluacion(EVALUACION)
desconocidas = [i for i in EV.index if i not in CAT.index]
if desconocidas:
    raise ValueError(f"Ids que no están en el catálogo: {desconocidas}. Revise que cada línea empiece con un id como A01 o M04.")
EV = CAT.loc[EV.index, ["opcion", "tipo", "reduce", "enfoque", "sector", "alerta_mala_adaptacion"]].join(EV)
EV["puntaje"] = ES.puntuar(EV, W).round(2)
EV["puntaje_pesos_iguales"] = ES.puntuar(EV, {k: 1 / len(ES.CRITERIOS) for k in ES.CRITERIOS}).round(2)
EV = EV.sort_values("puntaje", ascending=False)
EV["rango"] = range(1, len(EV) + 1)
EV["rango_pesos_iguales"] = EV.puntaje_pesos_iguales.rank(ascending=False, method="min").astype(int)
display(EV[["opcion", "tipo", *ES.CRITERIOS, "puntaje", "rango", "rango_pesos_iguales"]])
n5 = min(5, len(EV))
top = set(EV.index[:n5]); top_eq = set(EV.sort_values("puntaje_pesos_iguales", ascending=False).index[:n5])
ROBUSTEZ = len(top & top_eq) / max(1, n5)
print(f"Robustez: {len(top & top_eq)} de las {n5} primeras se mantienen con pesos iguales ({100 * ROBUSTEZ:.0f} %).")

AVISOS = []
n_m = EV.tipo.isin(["M", "MA"]).sum(); n_a = EV.tipo.isin(["A", "MA"]).sum(); n_mp = (EV.tipo == "M").sum()
if not 6 <= len(EV) <= 10: AVISOS.append(f"Evaluó {len(EV)} medidas; se piden entre 6 y 10.")
if n_a < 3: AVISOS.append(f"Solo {n_a} medidas de adaptación (A o MA); se piden al menos 3.")
if n_m < 2: AVISOS.append(f"Solo {n_m} medidas de mitigación (M o MA); se piden al menos 2.")
if n_mp < 1: AVISOS.append("No evaluó ninguna medida de mitigación pura (M): quite el # de al menos una línea M de la sección 5.")
am_ev = {a for i in EV.index for a in CAT.loc[i, "amenazas"]}
for a in AMEN_REL:
    if a not in am_ev: AVISOS.append(f"La amenaza «{RG.AMENAZA_CORTA[a]}» (riesgo medio o mayor) no tiene ninguna medida evaluada.")
f_ev = {f for i in EV.index for f in CAT.loc[i, "factores"]}
sin_f = [f for f in FACT if f not in f_ev]
if FACT and len(sin_f) == len(FACT): AVISOS.append("Ninguna medida atiende los indicadores críticos de la vulnerabilidad del cantón.")
sin_j = [i for i, r in EV.iterrows() if not str(r.justificacion).strip() or "justifique aquí" in str(r.justificacion)]
if sin_j: AVISOS.append(f"Falta la justificación de: {', '.join(sin_j)}.")
for i, r in EV.iterrows():
    if r.alerta_mala_adaptacion and r.mala_adaptacion >= 4:
        AVISOS.append(f"{i}: el catálogo advierte «{r.alerta_mala_adaptacion}»; explique por qué calificó riesgo bajo de mala adaptación.")
if not JUSTIFICACION_PESOS.strip(): AVISOS.append("Falta la justificación de los pesos.")
if POR_DEFECTO: AVISOS.append("Evaluación por defecto: llene la celda de la sección 6.")
print("\n" + ("Sin observaciones." if not AVISOS else "Revise antes de entregar:\n- " + "\n- ".join(AVISOS)))

## 8. Figuras de la matriz

- **a) Matriz:** cada fila es una medida y cada columna un criterio; a la derecha, el puntaje ponderado.
- **b) Efectividad frente a factibilidad:** arriba a la derecha están las medidas para **priorizar**; arriba a la izquierda, las que valen la pena pero necesitan **preparar condiciones** (normas, presupuesto, actores); abajo a la derecha, ganancias rápidas pero pequeñas. El tamaño del círculo son los cobeneficios.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(17, 0.55 * len(EV) + 3), gridspec_kw={"width_ratios": [1.5, 1], "wspace": 0.35})
ax = axs[0]
crit = list(ES.CRITERIOS)
ax.imshow(EV[crit].values, cmap="YlGn", vmin=1, vmax=5, aspect="auto")
ax.set_xticks(range(len(crit))); ax.set_xticklabels([c.replace("_", " ") for c in crit], rotation=30, ha="right")
etq = [f"{i} {o[:42]}{'…' if len(o) > 42 else ''} ({t})" for i, o, t in zip(EV.index, EV.opcion, EV.tipo)]
ax.set_yticks(range(len(EV))); ax.set_yticklabels(etq, fontsize=9.5)
for y in range(len(EV)):
    for x in range(len(crit)):
        v = int(EV[crit].iloc[y, x]); ax.text(x, y, v, ha="center", va="center", fontsize=10, color="white" if v >= 5 else "#222222")
for y, v in enumerate(EV.puntaje): ax.text(len(crit) - 0.3, y, f"{v:.2f}", va="center", fontsize=10.5, fontweight="bold")
ax.set_title("a) Matriz de evaluación (1–5) y puntaje ponderado", loc="left", fontweight="bold")
ax = axs[1]
COLT = {"M": "#C2531F", "A": "#2F6DB5", "MA": "#6A994E"}
ax.scatter(EV.factibilidad, EV.efectividad, s=60 * EV.cobeneficios, c=[COLT.get(t, "#888888") for t in EV.tipo], alpha=0.75, edgecolor="white")
for (fx, ef), g in EV.groupby(["factibilidad", "efectividad"]):
    ax.text(fx, ef + 0.16, "\n".join(g.index), ha="center", va="bottom", fontsize=10, linespacing=1.1)
ax.axhline(3, color="#AAAAAA", lw=0.8, ls="--"); ax.axvline(3, color="#AAAAAA", lw=0.8, ls="--")
ax.set_xlim(0.5, 5.5); ax.set_ylim(0.5, 5.5); ax.set_xlabel("factibilidad"); ax.set_ylabel("efectividad")
ax.text(5.4, 5.4, "priorizar", ha="right", va="top", color="#4E7A38", fontweight="bold")
ax.text(0.6, 5.4, "preparar condiciones", va="top", color="#6A7179"); ax.text(5.4, 0.6, "ganancias rápidas, pequeñas", ha="right", color="#6A7179")
for t, c in COLT.items(): ax.scatter([], [], c=c, s=80, label={"M": "mitigación", "A": "adaptación", "MA": "las dos"}[t])
ax.legend(loc="lower left", frameon=False, title="tamaño = cobeneficios")
ax.set_title("b) Efectividad frente a factibilidad", loc="left", fontweight="bold")
fig.suptitle(f"Medidas para {FOCO} ({PROVINCIA})" + ("  ·  EVALUACIÓN POR DEFECTO" if POR_DEFECTO else ""), x=0.01, ha="left", fontsize=15, fontweight="bold")
fig.savefig(RES / f"RD2_03_matriz_{SP}.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## 9. Marco normativo: dónde encaja cada medida

Cada medida se ubica en los instrumentos que la respaldan (tema 2.1 de la unidad):

- **Acuerdo de París y Segunda NDC 2026–2035** (CMNUCC): compromisos de mitigación y de adaptación del Ecuador.
- **PLANACC** (Plan Nacional de Adaptación): sectores priorizados de la adaptación.
- **Marco de Sendai 2015–2030**: sus cuatro prioridades de acción para reducir el riesgo de desastres.
- **PDOT** del cantón: donde la medida se vuelve programa, presupuesto y ordenanza.

El cuaderno sugiere la prioridad de Sendai y un **actor responsable**. Si el equipo sabe quién lo haría en su cantón, corríjalo abajo (una línea por medida: `id; actor`).

In [ ]:
# @title Actores corregidos por el equipo (opcional)
ACTORES_EQUIPO = """
"""
# Ejemplo:  A01; GAD de Chone (Unidad de Gestión de Riesgos) con la SNGR, coordinación zonal 4

_act = {}
for l in ACTORES_EQUIPO.strip().splitlines():
    if ";" in l and not l.strip().startswith("#"):
        k, v = l.split(";", 1); _act[k.strip()] = v.strip()
MARCO = pd.DataFrame({
    "opcion": EV.opcion, "tipo": EV.tipo, "reduce": EV.reduce, "NDC / PLANACC (sector)": EV.sector,
    "Sendai": [VU.SENDAI.get(VU.SENDAI_MEDIDA.get(i), "— (mitigación: meta de la NDC y del Acuerdo de París)") for i in EV.index],
    "actor": [_act.get(i, VU.actor(s)) for i, s in zip(EV.index, EV.sector)]}, index=EV.index)
display(MARCO)
print("Prioridades de Sendai cubiertas:", sorted({s[:1] for s in MARCO.Sendai if s[0].isdigit()}) or "ninguna")

## 10. Cartera priorizada y riesgo residual

La **cartera** son las 6 mejores medidas por puntaje, con al menos 2 de mitigación y 2 de adaptación.

El **riesgo residual** es el riesgo que queda después de aplicar la cartera. Con un supuesto simple, cada medida de adaptación mejora en `EFECTO` puntos los indicadores que atiende (baja la sensibilidad o sube la capacidad); el ordenamiento territorial (A03) y la reubicación (A06) bajan la exposición. **Las medidas de mitigación pura no cambian el riesgo local en el corto plazo**: su efecto es global y a largo plazo. Esa diferencia es la que debe discutir en el capítulo.

In [ ]:
# @title Supuesto del efecto de cada medida
EFECTO = 1  # @param {type:"slider", min:1, max:2, step:1}

cartera, m, a = [], 0, 0
for i, r in EV.iterrows():
    if len(cartera) >= 6: break
    es_m, es_a = r.tipo in ("M", "MA"), r.tipo in ("A", "MA")
    faltan_m, faltan_a = max(0, 2 - m), max(0, 2 - a)
    if 6 - len(cartera) <= faltan_m + faltan_a and not ((es_m and faltan_m) or (es_a and faltan_a)):
        continue
    cartera.append(i); m += es_m; a += es_a
CARTERA = EV.loc[cartera, ["opcion", "tipo", "reduce", "puntaje", "alerta_mala_adaptacion", "justificacion"]].join(MARCO[["Sendai", "actor"]])
CARTERA["responde_a"] = [SUG.motivo.get(i, "mitigación de la provincia") for i in cartera]
display(CARTERA)

EXPO_F = PUNT.set_index("canton").loc[FOCO, list(A)].to_dict()
RESID, CAMBIOS = VU.efecto_cartera(C_FOCO, EXPO_F, A, CAT.loc[cartera], EFECTO)
display(RESID[["amenaza", "R_antes", "clase_antes", "R_despues", "clase_despues", "reduccion_%", "V_antes", "V_despues", "E_antes", "E_despues"]])
print("Cambios supuestos:\n- " + "\n- ".join(CAMBIOS))

fig, ax = plt.subplots(figsize=(1.8 * len(RESID) + 4, 4.4))
x = np.arange(len(RESID))
b0 = ax.bar(x - 0.2, RESID.R_antes, 0.4, color=[RG.COLOR_CLASE[c] for c in RESID.clase_antes], edgecolor="#333333", label="antes")
b1 = ax.bar(x + 0.2, RESID.R_despues, 0.4, color=[RG.COLOR_CLASE[c] for c in RESID.clase_despues], edgecolor="#333333", hatch="//", label="con la cartera")
ax.bar_label(b0, labels=[f"{v:.1f}\n{c}" for v, c in zip(RESID.R_antes, RESID.clase_antes)], fontsize=9)
ax.bar_label(b1, labels=[f"{v:.1f}\n{c}" for v, c in zip(RESID.R_despues, RESID.clase_despues)], fontsize=9)
ax.set_xticks(x, RESID.amenaza); ax.set_ylim(0, 5.9); ax.set_ylabel("Riesgo (1–5)"); ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"{FOCO}: riesgo antes y después de la cartera (supuesto: {EFECTO} punto por medida)", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(RES / f"RD2_03_riesgo_residual_{SP}.png", dpi=200); plt.show()

## 11. Guardar

In [ ]:
EV.to_csv(RES / f"RD2_03_evaluacion_{SP}.csv", encoding="utf-8")
CARTERA.to_csv(RES / f"RD2_03_cartera_{SP}.csv", encoding="utf-8")
MARCO.to_csv(RES / f"RD2_03_marco_normativo_{SP}.csv", encoding="utf-8")
RESID.to_csv(RES / f"RD2_03_riesgo_residual_{SP}.csv", index=False, encoding="utf-8")
meta = {"material": "CC-IRD-P2-03 v1.0", "equipo": ESTUDIANTES, "provincia": PROVINCIA, "canton_foco": FOCO,
        "canton_foco_elegido_por_equipo": bool(CANTON_FOCO.strip()), "amenazas_relevantes": AMEN_REL, "factores_criticos": FACT,
        "pesos": W, "justificacion_pesos": JUSTIFICACION_PESOS, "robustez_top5": ROBUSTEZ, "evaluacion_por_defecto": POR_DEFECTO,
        "rd2_02_provisional": bool(META2.get("provisional")), "cartera": cartera, "efecto_supuesto": EFECTO,
        "cambios_supuestos": CAMBIOS, "avisos": AVISOS, "fecha": datetime.datetime.now().isoformat(timespec="seconds")}
(RES / "RD2_03_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Guardado en", RES)
for p in sorted(RES.glob("RD2_03_*")): print("  ", p.name)

## 12. Preguntas para el capítulo 2 (sección 4, «Medidas»)

1. ¿Qué medidas evaluó y a qué responde cada una (amenaza o indicador crítico del cantón)? ¿Por qué eligió esa medida de mitigación pura?
2. ¿Cuáles quedaron en la cartera? ¿Su ranking es robusto con pesos iguales? Si cambia, ¿qué criterio lo explica y cómo justifica sus pesos?
3. Compare una medida de adaptación y una de mitigación de su matriz: **qué reduce cada una, a qué escala, en cuánto tiempo y quién se beneficia**.
4. ¿Cuánto baja el riesgo del cantón con la cartera (riesgo residual)? ¿Por qué no llega a «muy bajo»? ¿Qué parte del riesgo queda y qué haría con ella (transferirla con seguros, prepararse para responder)?
5. ¿Qué medida tiene más riesgo de mala adaptación y cómo lo evitaría (diseño, lugar, a quién beneficia o perjudica aguas abajo)?
6. ¿En qué instrumentos encaja la cartera (Segunda NDC, PLANACC, Sendai, PDOT)? ¿Qué prioridad de Sendai queda sin cubrir y qué actor del cantón tendría que liderar la primera medida?

**Para el informe:** la tabla de evaluación con sus justificaciones (puede ir en un anexo), los pesos y su justificación, la prueba de robustez, la cartera, la tabla del marco normativo y la figura del riesgo residual con su supuesto escrito.

**Citas.** IPCC (2022). *Climate Change 2022: Impacts, Adaptation and Vulnerability*, GTII, cap. 12 (América Central y del Sur) y cap. 17 (opciones de adaptación) · IPCC (2022). *Climate Change 2022: Mitigation of Climate Change*, GTIII · República del Ecuador (2025). *Segunda Contribución Determinada a Nivel Nacional 2026–2035* · MAATE (2023). *Plan Nacional de Adaptación al Cambio Climático del Ecuador (2022–2027)* · UNDRR (2015). *Marco de Sendai para la Reducción del Riesgo de Desastres 2015–2030* · Belton, V. y Stewart, T. (2002). *Multiple Criteria Decision Analysis*. Springer.